```markdown
# Step 1: Understanding BERT and XLM-RoBERTa

### BERT (Bidirectional Encoder Representations from Transformers)
* **Architecture**: Based on the Transformer encoder architecture.
* **Training**: Pre-trained using Masked Language Modeling (MLM) and Next Sentence Prediction (NSP).
* **Language**: Primarily monolingual (e.g., English), though multilingual versions exist (mBERT).

### XLM-RoBERTa (Cross-lingual Language Model - Robustly Optimized BERT Approach)
* **Architecture**: An extension of RoBERTa trained on a massive multilingual dataset (CommonCrawl).
* **Training**: Uses only the Masked Language Modeling (MLM) objective, discarding NSP.
* **Language**: Excellent cross-lingual performance across over 100 languages.

### Tokenization
Both models process text by breaking down strings into subword tokens, mapping them to vocabulary indices (`input_ids`), and outputting masks to distinguish content from padding.
```

In [6]:
# Install transformers library if not already installed
!pip install -q transformers

```markdown
# Step 2: Tokenizing Text

Here we instantiate the tokenizers and experiment with single and pairwise text tokenization using `encode_plus` and `decode`.

In [7]:
from transformers import BertTokenizer, XLMRobertaTokenizer

# Instantiate tokenizers
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlm_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

sentence_1 = "Natural Language Processing is fascinating."
sentence_2 = "We can represent text as numbers."

print("--- BERT Single Sentence ---")
# Calling the tokenizer directly serves as the standard, robust encoding pipeline (equivalent to encode_plus)
bert_encoded = bert_tokenizer(
    sentence_1,
    add_special_tokens=True,
    return_token_type_ids=True,
    return_attention_mask=True
)
print("Input IDs:      ", bert_encoded['input_ids'])
print("Token Type IDs: ", bert_encoded['token_type_ids'])
print("Attention Mask: ", bert_encoded['attention_mask'])
print("Decoded:        ", bert_tokenizer.decode(bert_encoded['input_ids']))

print("\n--- XLM-RoBERTa Two Sentences ---")
xlm_encoded = xlm_tokenizer(
    sentence_1,
    sentence_2,
    add_special_tokens=True
)
print("Input IDs:      ", xlm_encoded['input_ids'])
print("Decoded:        ", xlm_tokenizer.decode(xlm_encoded['input_ids']))

--- BERT Single Sentence ---
Input IDs:       [101, 3019, 2653, 6364, 2003, 17160, 1012, 102]
Token Type IDs:  [0, 0, 0, 0, 0, 0, 0, 0]
Attention Mask:  [1, 1, 1, 1, 1, 1, 1, 1]
Decoded:         [CLS] natural language processing is fascinating. [SEP]

--- XLM-RoBERTa Two Sentences ---
Input IDs:       [0, 24955, 83658, 123996, 214, 83, 102919, 1916, 5, 2, 2, 1401, 831, 33636, 7986, 237, 101935, 5, 2]
Decoded:         <s> Natural Language Processing is fascinating.</s></s> We can represent text as numbers.</s>


In [8]:
print("\n--- XLM-RoBERTa Two Sentences ---")
xlm_encoded = xlm_tokenizer(
    sentence_1,
    sentence_2,
    add_special_tokens=True
)
print("Input IDs:      ", xlm_encoded['input_ids'])
print("Decoded:        ", xlm_tokenizer.decode(xlm_encoded['input_ids']))


--- XLM-RoBERTa Two Sentences ---
Input IDs:       [0, 24955, 83658, 123996, 214, 83, 102919, 1916, 5, 2, 2, 1401, 831, 33636, 7986, 237, 101935, 5, 2]
Decoded:         <s> Natural Language Processing is fascinating.</s></s> We can represent text as numbers.</s>


```markdown
# Step 3: Preparing Input Data for the Model

We ensure input sequences are of uniform length using padding and truncation, and check the vocab sizes and special token configurations.

In [9]:
max_len = 16

# Call the tokenizer directly instead of encode_plus to avoid lookup attribute errors
prepared_inputs = xlm_tokenizer(
    sentence_1,
    max_length=max_len,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    add_special_tokens=True
)

print("Input IDs:", prepared_inputs['input_ids'])
print("Attention Mask:", prepared_inputs['attention_mask'])
print("\nSpecial Tokens Map:", xlm_tokenizer.special_tokens_map)
print("Vocab Size:", xlm_tokenizer.vocab_size)

Input IDs: [0, 24955, 83658, 123996, 214, 83, 102919, 1916, 5, 2, 1, 1, 1, 1, 1, 1]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

Special Tokens Map: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}
Vocab Size: 250002


```markdown
# Step 4: Loading and Exploring the Dataset

We will read the sample dataset available in the runtime environment (`/content/sample_submission.csv` or standard housing data) to demonstrate data ingestion.

In [10]:
import pandas as pd

# Load the actual training dataset
df = pd.read_csv('/content/train.csv')

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (12120, 6)


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


In [11]:
# Removed sample_submission.csv loading to prevent overwriting the real train.csv dataset
print("Using real train.csv dataset loaded in df.")

Using real train.csv dataset loaded in df.


```markdown
# Step 5: Creating Cross-Validation Folds

Using `StratifiedKFold` to create 5 distinct splits of the data while maintaining class balance.

In [12]:
import numpy as np
from sklearn.model_selection import StratifiedKFold

# Prepare inputs from the actual NLI dataset ('premise' and 'hypothesis')
X_premise = df['premise'].values
X_hypothesis = df['hypothesis'].values
y = df['label'].values

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

folds = []
for fold, (train_idx, val_idx) in enumerate(kf.split(X=df, y=y)):
    train_split = df.iloc[train_idx]
    val_split = df.iloc[val_idx]
    folds.append((train_split, val_split))
    print(f"Fold {fold + 1} | Train size: {len(train_split)} | Val size: {len(val_split)} | Val Label Mean: {val_split['label'].mean():.2f}")

Fold 1 | Train size: 9696 | Val size: 2424 | Val Label Mean: 0.99
Fold 2 | Train size: 9696 | Val size: 2424 | Val Label Mean: 0.99
Fold 3 | Train size: 9696 | Val size: 2424 | Val Label Mean: 0.99
Fold 4 | Train size: 9696 | Val size: 2424 | Val Label Mean: 0.99
Fold 5 | Train size: 9696 | Val size: 2424 | Val Label Mean: 0.99


# Step 6: PyTorch Dataset, Model Ingestion & Training Loop

We define a custom PyTorch Dataset, instantiate our model using `AutoModelForSequenceClassification`, and write a training loop to optimize the weights on a selected fold.

In [13]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

class TextClassificationDataset(Dataset):
    def __init__(self, premises, hypotheses, labels, tokenizer, max_len=128):
        self.premises = premises
        self.hypotheses = hypotheses
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.premises)

    def __getitem__(self, idx):
        premise = str(self.premises[idx])
        hypothesis = str(self.hypotheses[idx])
        label = self.labels[idx]

        # Encode premise and hypothesis pairwise
        encoding = self.tokenizer(
            premise,
            hypothesis,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [14]:
# Setup training configuration
MAX_LEN = 64
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Get Fold 1 split data
train_df, val_df = folds[0]

# Initialize Dataset with premise and hypothesis columns
train_dataset = TextClassificationDataset(
    premises=train_df['premise'].values,
    hypotheses=train_df['hypothesis'].values,
    labels=train_df['label'].values,
    tokenizer=xlm_tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextClassificationDataset(
    premises=val_df['premise'].values,
    hypotheses=val_df['hypothesis'].values,
    labels=val_df['label'].values,
    tokenizer=xlm_tokenizer,
    max_len=MAX_LEN
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

In [15]:
# Load Model with the correct number of labels (3 classes for NLI dataset: entailment, neutral, contradiction)
model = AutoModelForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=3)
model = model.to(DEVICE)

# Optimizer and Learning Rate Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
# Training and Evaluation Loop
for epoch in range(EPOCHS):
    print(f"\n=== Epoch {epoch + 1} / {EPOCHS} ===")

    # Training
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_train_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Average Training Loss: {avg_train_loss:.4f}")

    # Validation
    model.eval()
    total_val_loss = 0
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_val_loss += loss.item()

            logits = outputs.logits.detach().cpu().numpy()
            label_ids = labels.to('cpu').numpy()

            predictions.extend(np.argmax(logits, axis=1))
            true_labels.extend(label_ids)

    avg_val_loss = total_val_loss / len(val_loader)
    val_accuracy = accuracy_score(true_labels, predictions)
    val_f1 = f1_score(true_labels, predictions, average='macro')

    print(f"Validation Loss: {avg_val_loss:.4f} | Accuracy: {val_accuracy:.4f} | Macro F1: {val_f1:.4f}")


=== Epoch 1 / 3 ===
Average Training Loss: 1.0379
Validation Loss: 0.8758 | Accuracy: 0.6184 | Macro F1: 0.6158

=== Epoch 2 / 3 ===
Average Training Loss: 0.7874
Validation Loss: 0.7579 | Accuracy: 0.6844 | Macro F1: 0.6834

=== Epoch 3 / 3 ===
Average Training Loss: 0.6093
Validation Loss: 0.7861 | Accuracy: 0.6877 | Macro F1: 0.6875


In [17]:
### Note on Cross-Validation
#We are utilizing the real StratifiedKFold split defined in **Step 5** on the authentic `premise` and `hypothesis` columns instead of dummy placeholders.